# Проблемы:
* ~~jrn.dep_id IN (SELECT UNNEST(STRING_TO_ARRAY(units, ','))::INT) - Проблемы с подзапросом фильтром~~
* **IN (SELECT id from table) - Добавить IN как игнор** (?)
* ~~Добавить в sqlfluff проверку "*" - Не правильно выходит логика в конце~~
* ~~Исправить alias - Должен обизательно быть alias в блоке from и в join - нах~~
* Исправление on true в join
* Исправление других типов join (cross, outter и literal)
* Надо добавить проверку на cte внутри cte 

In [12]:
from validator.validator import ValidationCheker

In [13]:
sql = """
    select
        p.id
    from
        base_info.profile p
    inner join
        (select
            s1.ticker,
            s1.p,
            s1.s
        from
            base_info.symbols as s1) as s
        on p.display_symbol = s.ticker
        and (p.d = s.p or p.r = s.s)
"""

In [14]:
cheker = ValidationCheker()

cheker.check(sql)

[]

In [2]:
file = 'test\\test.sql'

with open('functions\\test.sql', 'r', encoding='utf-8') as file:
    sql = file.read()

FileNotFoundError: [Errno 2] No such file or directory: 'functions\\test.sql'

In [29]:
import sqlglot
from sqlglot import exp
import json
import os
from dataclasses import dataclass, field, asdict

@dataclass
class Tables:
    type: str
    alias: str
    level: str
    name: str
    global_path: str

@dataclass
class Mains:
    tables: dict[str, Tables] = field(default_factory=dict)
    joins: list = field(default_factory=list)

@dataclass
class SelectBlock:
    block_name: str
    tables: dict[str, Tables] = field(default_factory=dict)
    joins: list = field(default_factory=list)
    column_lineage: dict = field(default_factory=dict)

@dataclass
class Objects:
    alias_name: str
    obj_type: str
    parent: str
    select_blocks: list[SelectBlock] = field(default_factory=list)

@dataclass
class SemanticJSON:
    metadata: dict = field(default_factory=dict) # Думаю буду использовать Jira/Confluence
    objects: dict[str, Objects] = field(default_factory=dict)
    mains: dict[str, Mains] = field(default_factory=dict)

In [ ]:
class SemanticExtractor:
    
    def extract(self, sql: str, save_path ='output\\data.json'):
        self.json_path = save_path
        
        ast = sqlglot.parse_one(sql, dialect="postgres") # AST(Abstract  Syntax Tree) - это дерево которое полностью повторяет SQL сиентакс
                
        semantic = SemanticJSON()

        other_query = ast.copy()
        ast_other = other_query.args.get("with_")

        print(semantic)
        
        print(ast_other)

        if ast_other:
            self.extract_cte(ast_other, semantic)

        main_query = ast.copy()
        main_query.args.pop("with_", None) # Удаляем все кроме основного запроса
        level = 'main'

        
        
    def merge_relationships(self, new_relationships):

        if not os.path.exists(self.json_path):
            with open(self.json_path, "w") as f:
                json.dump({}, f, indent=4)
        
        try:
            with open(self.json_path) as f:
                data = json.load(f)
        except (json.JSONDecodeError, FileNotFoundError):
            data = {}

        existing = set(data.keys())

        for key, rel in new_relationships.items():

            if key not in existing:
                data[key] = rel
                existing.add(key)

        with open(self.json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    
    
    def lower_case(self, t):
        return str(t).lower()

    def extract_union_selects(self, node):
        if isinstance(node, exp.Union):
            yield from self.extract_union_selects(node.this)
            yield from self.extract_union_selects(node.expression)
        else:
            yield node
    
    def get_join_source(self, node):
                
        # CTE
        if isinstance(node, exp.Table) and node.db == '':

            return {
                "type": "cte",
                "name": self.lower_case(node.name),
                "alias": node.alias_or_name
            }

        # Обычная таблица
        if isinstance(node, exp.Table):

            return {
                "type": "table",
                "name": ".".join(
                    x for x in [
                        self.lower_case(node.catalog),
                        self.lower_case(node.db),
                        self.lower_case(node.name)
                    ]
                    if x
                ),
                "alias": node.alias_or_name
            }

        # Подзапрос
        if isinstance(node, exp.Subquery):

            return {
                "type": "subquery",
                "alias": node.alias_or_name
            }

        return {
            "type": type(node).__name__
        }
        
    # Вывести таблицы
    def extract_tables(self, ast, semantic, ttype, level):

        if ttype == 'subquery':
            alias_name = ast.alias
            semantic.tables[alias_name] = {
                "type": ttype,
                "alias": alias_name,
                "level": level,
                "name": f"{level}-{ttype}-{alias_name}",
                "global_path": f"{level}-{ttype}-{alias_name}"
            }
        elif ttype == 'cte':
            alias_name = ast.alias
            name = ast.name
            semantic.tables[alias_name] = {
                "type": ttype,
                "alias": alias_name,
                "level": level,
                "name": f"main-{ttype}-{name}",
                "global_path": f"main-{ttype}-{name}"
            }
        elif ttype == 'table':
            for table in ast.find_all(exp.Table):
                alias_name = table.alias_or_name
                semantic.tables[alias_name] = {
                    "type": ttype,
                    "alias": table.alias,
                    "level": level,
                    "name": ".".join(
                        x for x in [table.catalog, self.lower_case(table.db), self.lower_case(table.name)] if x
                    ),
                    "global_path": f"{level}"
                }

    def get_type_of_table(self, ast, semantic):
        ttype = ''

        if isinstance(ast, exp.Table) and ast.db == '':
            ttype = 'cte'
        elif isinstance(ast, exp.Table) and ast.db != '':
            ttype = 'table'
        elif isinstance(ast, exp.Subquery):
            ttype = 'subquery'

        return ttype

    def extract_from(self, ast, semantic, level):
        ast_from = ast.find(exp.From)

        if ast_from is None:
            print(f"[WARNING] FROM не найден: {type(ast).__name__}")
            print(ast)
            return

        ast_from = ast_from.this

        if ast_from is None:
            print(f"[WARNING] FROM.this пустой: {type(ast).__name__}")
            print(ast)
            return
        
        ttype = self.get_type_of_table(ast_from, semantic)

        print(ttype)

        self.extract_tables(ast_from, semantic, ttype, level)

        

    
    # Вывести join-ы        
    def is_join_key(self, node):
        return (
            isinstance(node.left, exp.Column)
            and isinstance(node.right, exp.Column)
        )

    def extract_operand(self, node, semantic, level):

        if isinstance(node, exp.Column):
            if semantic.tables[node.table]:
                i = semantic.tables[node.table]
                return {
                    "table": i["name"],
                    "column": node.name
                }

        if isinstance(node, exp.Literal):
            return {
                "literal": node.this
            }

        return {
            "expression": node.sql()
        }
    
    
    def extract_conditions(self, node, semantic, level):
        """
        Возвращает список всех элементарных условий из ON.
        """

        if node is None:
            return []

        if isinstance(node, bool):
            return [{
                "expression": str(node).upper(),
                "is_join_key": False
            }]


        # Убираем скобки
        if isinstance(node, exp.Paren):
            return self.extract_conditions(
                node.this,
                semantic,
                level
            )

        # Разбираем AND
        if isinstance(node, exp.And):
            return (
                self.extract_conditions(node.left, semantic, level)
                + self.extract_conditions(node.right, semantic, level)
            )

        # Разбираем OR
        if isinstance(node, exp.Or):
            return [{
                "operator": "OR",
                "conditions": (
                    self.extract_conditions(node.left, semantic, level)
                    + self.extract_conditions(node.right, semantic, level)
                )
            }]

        # Игнорируем IN
        if isinstance(node, exp.In):
            return []

        # Простое сравнение
        if isinstance(node, (
            exp.EQ,
            exp.GT,
            exp.GTE,
            exp.LT,
            exp.LTE,
            exp.NEQ,
        )):
            
            return [{
                "operator": node.key,
                "left": self.extract_operand(node.left, semantic, level),
                "right": self.extract_operand(node.right, semantic, level),
                "is_join_key": self.is_join_key(node)
            }]

        return [{
            "expression": node.sql()
        }]
    
    
    def extract_joins(self, ast, semantic, level):
        select = ast.find(exp.Select)
        joins = select.args.get("joins", [])

        print('JOINS:')

        for join in joins:

            print(join)

            on = join.args.get("on")
            print(on.this)

            ttype = self.get_type_of_table(join.this, semantic)
            self.extract_tables(join.this, semantic, ttype, level)
            
            conditions = self.extract_conditions(join.args.get("on").this, semantic, level)
            
            
            join_keys = [
                {k: v for k, v in c.items() if k != "is_join_key"}
                for c in conditions
                if c.get("is_join_key")
            ]

        
            if join_keys:
                join_info = {
                    "type": self.lower_case(join.args.get("side", "INNER")),
                    "table": self.get_join_source(
                        join.this
                    ),
                    "join_keys": join_keys
                }

                semantic.joins.append(join_info)
            
    def resolve_expression(self, node, semantic, level):

        columns = []

        for col in node.find_all(exp.Column): 

            if semantic.tables[col.table]:
                i = semantic.tables[col.table]
                columns.append({
                    "level": level,
                    "table": i["name"],
                    "column": col.name
                })

        return columns
    
            
    def extract_columns(self, ast, semantic, alias, level):

        select = ast.find(exp.Select)

        if not select:
            return

        for expression in select.expressions:

            output_name = expression.alias_or_name

            semantic.column_lineage[f"{alias}.{output_name}"] = self.resolve_expression(expression, semantic, level)

    def from_block_check(self, ast):
        ast_from = ast.find(exp.From)
        
        if ast_from is None:
            print(f"[WARNING] FROM не найден: {type(ast).__name__}")
            print(ast)
            return 0

        ast_from = ast_from.this

        if ast_from is None:
            print(f"[WARNING] FROM.this пустой: {type(ast).__name__}")
            print(ast)
            return 0

    def extract_subquery(self, ast, semantic, level):
        subqueries = list(ast.find_all(exp.Subquery, bfs=False))
            
        if not subqueries:
            print("Subquery нет")
            return

        for subquery in subqueries:

            if self.from_block_check(subquery.this) == 0:
                continue
            
            key = f"{level}-subquery-{subquery.alias}"
            semantic.objects[key] = Objects(alias_name=subquery.alias, obj_type='subquery', parent=level)

            selects = list(self.extract_union_selects(subquery.this))

            for i in range(len(selects)):
                name = f'main_{i+1}'
                query = selects[i]
                subquery_level = f"{key}:{name}"

                print(query)

                semantic.objects[key].select_blocks.append(SelectBlock(block_name=name))
            
                self.extract_from(query, semantic.objects[key].select_blocks[i], subquery_level)

                self.extract_joins(
                    query,
                    semantic.objects[key].select_blocks[i],
                    subquery_level
                )
                
                self.extract_columns(
                    query,
                    semantic.objects[key].select_blocks[i],
                    query.alias,
                    subquery_level
                )
                
                self.extract_subquery(
                    query,
                    semantic,
                    subquery_level,
                )

    

    def extract_cte(self, ast, semantic, level=None):
        # print(ast)

        ctes = list(ast.find_all(exp.CTE, bfs=False))
    
        if not ctes:
            print("CTE нет")
            return

        for cte in ctes:

            key = f"main-cte-{cte.alias}"
            semantic.objects[key] = Objects(alias_name=cte.alias, obj_type='cte', parent='main')

            selects = list(self.extract_union_selects(cte.this))

            print(len(selects))
                    
            for i in range(len(selects)):
                name = f'main_{i+1}'
                query = selects[i]
                cte_level = f"{key}:{name}"

                query.set("where", None)

                semantic.objects[key].select_blocks.append(SelectBlock(block_name=name))

                self.extract_from(query, semantic.objects[key].select_blocks[i], cte_level)

                self.extract_joins(
                                query,
                                semantic.objects[key].select_blocks[i],
                                cte_level
                            )

                self.extract_columns(
                    query,
                    semantic.objects[key].select_blocks[i],
                    query.alias,
                    cte_level
                )
                
                self.extract_subquery(
                    query,
                    semantic,
                    cte_level
                )
                
                self.extract_cte(
                    query,
                    semantic,
                    cte_level
                )

            print(semantic.objects[key])

        
    
    # ==============================================================================================================================
    
    # =====================================================================
    # RELATIONSHIPS
    # =====================================================================

    def get_source_type(
        self,
        source_name,
        semantic
    ):
        """
        Определяет тип источника.

        Если source_name найден в semantic.objects,
        значит это CTE или Subquery.

        Если не найден — это физическая таблица.

        Например:

            main-cte-main
                -> cte

            main-cte-table_1
                -> cte

            main_1-subquery-m2
                -> subquery

            cbs.main
                -> table
        """

        obj = semantic.objects.get(
            source_name
        )

        if obj is not None:
            return obj.obj_type

        return "table"

In [31]:
SemanticExtractor().extract(sql)

SemanticJSON(metadata={}, objects={}, mains={})
WITH rbt_mfo AS (SELECT DISTINCT ON (rbt_mfo1.asbuka_id) rbt_mfo1.asbuka_id, rbt_mfo1.name_ru, rbt_mfo1.e_business_risk_category_id FROM rdm.r_business_type AS rbt_mfo1 WHERE 1 = 1 AND rbt_mfo1.arh = 0), pay_cash AS (SELECT jrn.ord_id, jrn.dep_id, CAST(jrn.doper AS DATE) /* дата проведения чдп/пдп */, ca.fio, CASE WHEN det.cvalue = 'WSHD_FULL' THEN 'ПДП' WHEN det.cvalue IN ('WSHD_DEC_EQUAL', 'WSHD_DEC_PERIOD') THEN 'ЧДП' END AS payment_type, ca.code FROM cbs.t_operjrn AS jrn INNER JOIN cbs.t_operdet AS det ON det.id = jrn.id AND det.njrn = jrn.njrn AND jrn.dep_id IN (SELECT CAST(EXPLODE(STRING_TO_ARRAY(units, ',')) AS INT)) LEFT JOIN cbs.t_scen_std AS tss ON tss.id = jrn.bop_id AND tss.nord = jrn.noper LEFT JOIN cash AS ca ON ca.id = jrn.id AND ca.njrn + 1 = jrn.njrn WHERE 1 = 1 AND CAST(jrn.doper AS DATE) >= s_date AND CAST(jrn.doper AS DATE) <= e_date AND det.cvalue IN ('WSHD_FULL' /* ПДП */, 'WSHD_DEC_EQUAL' /* 'ЧДП с уменьшением ежеме